<a href="https://colab.research.google.com/github/mehmetbozdemir24/Magibu/blob/main/02_Model_FineTuning/Qwen3_5_4b_fine_tuned_model_with_food_recipes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Drive Bağlantısı ve HF Login ---
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub datasets

# Hugging Face'e giriş yapıyoruz
from huggingface_hub import notebook_login
notebook_login()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%capture
# --- Hücre 1: Kütüphane Kurulumları ---
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# --- Hücre 2: Veri Setini HF Hub'a Yükleme ---
from datasets import load_dataset

# 1. Veri setini yerel Drive'ından yüklüyoruz
file_path = "/content/drive/MyDrive/Colab Notebooks/Magibu/data/processed/recipe_dataset.jsonl"
dataset = load_dataset("json", data_files=file_path, split="train")

# 2. HUGGING FACE KULLANICI ADINI BURAYA YAZ:
DATASET_REPO_ID = "nypgd/nefis-yemek-tarifleri-dataset"

# 3. Veri setini doğrudan Hugging Face profiline yüklüyoruz
print("Veri seti Hugging Face'e yükleniyor...")
dataset.push_to_hub(DATASET_REPO_ID)
print(f"🎉 Veri seti başarıyla {DATASET_REPO_ID} adresine yüklendi!")

Generating train split: 0 examples [00:00, ? examples/s]

Veri seti Hugging Face'e yükleniyor...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  80%|#######9  |  526kB /  659kB            

🎉 Veri seti başarıyla nypgd/nefis-yemek-tarifleri-dataset adresine yüklendi!


In [ ]:
# --- Hücre 3: Model Yükleme ---
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = True, # RAM/VRAM tasarrufu için 4bit yüklüyoruz
    use_gradient_checkpointing = "unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [ ]:
# --- Hücre 4: LoRA Ayarları ---
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# --- Hücre 5: Veri Formatlama ---
def convert_to_vision_format(sample):
    """
    Kendi JSONL yapındaki mesajları Unsloth Vision formatına çevirir.
    Senin verinde resim (image: null) olmadığı için sadece text objesi oluşturuyoruz.
    """
    conversation = []
    for msg in sample["messages"]:
        role = msg["role"]
        text_content = msg["content"]

        # Vision model "content" alanını bir liste olarak bekler
        formatted_content = [
            {"type": "text", "text": text_content}
        ]

        conversation.append({
            "role": role,
            "content": formatted_content
        })

    return { "messages" : conversation }

# Dönüşümü tüm veri setine uygula
converted_dataset = [convert_to_vision_format(sample) for sample in dataset]

# İlk örneğin nasıl dönüştüğüne bakalım
print(converted_dataset[0])

{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Cold Brew Tarifi nasıl yapılır? Tarifini verebilir misin?'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Tabii, işte nefis bir Cold Brew Tarifi tarifi!\n\n⏱️ **Süre:** PT5M | 🍽️ **Porsiyon:** 2\n\n**Malzemeler:**\n- 8 yemek kaşığı kalın öğütülmüş kahve (80 g)\n- 750 ml su\n\n**Yapılışı:**\n1. Kalın öğütülmüş kahveyi French pressin haznesine alalım.\n2. Üzerine soğuk ya da oda sıcaklığındaki suyu ekleyerek 1-2 kez karıştıralım ve kapağını sıkıca kapatalım.\n3. Bu şekilde buzdolabına kaldıralım ve 10 saat boyunca dinlenmeye bırakalım.\n4. Sürenin sonunda French pressin filtresini yavaşça aşağıya doğru bastırarak kahve tanelerini ayıralım.\n5. Ardından bir şişeye boşaltarak buzdolabında saklayabilir, isterseniz bol buzlu bardakta servis edebilirsiniz. Afiyet olsun!\n\nŞimdiden afiyet olsun!'}]}]}


In [ ]:
# --- Hücre 6: Eğitimsiz Modeli Test Etme ---
FastVisionModel.for_inference(model) # Çıkarım (Inference) modunu aktif et

# Veri setindeki ilk örneğin sorusunu çekiyoruz
ornek_soru = converted_dataset[0]["messages"][0]["content"][0]["text"]

print(f"❓ Sorulan Soru: {ornek_soru}\n")
print("🤖 --- Eğitimsiz Modelin Çıktısı ---")

messages = [
    {"role": "user", "content": [
        {"type": "text", "text": ornek_soru}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    text=input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# Çıktıyı üret
_ = model.generate(**inputs,
                   streamer=text_streamer,
                   max_new_tokens=2048,      # Sınırı makul bir seviyeye çektik
                   use_cache=True,
                   temperature=0.7,          # Daha odaklı ve az rastgele cevap vermesi için
                   min_p=0.1,
                   repetition_penalty=1.18)  # Sonsuz döngüye girmesini engelleyen ceza

❓ Sorulan Soru: Cold Brew Tarifi nasıl yapılır? Tarifini verebilir misin?

🤖 --- Eğitimsiz Modelin Çıktısı ---
Here's a thinking process that leads to the suggested cold brew recipe:

1.  **Analyze the Request:**
    *   **Topic:** Cold Brew Coffee (Sıcak Kahve değil, soğuk demli kahve).
    *   **Language:** Turkish ("Tarifi nasıl yapılır?", "Tarifini verebilir misim?").
    *   **Goal:** Provide a clear, easy-to-follow recipe for making cold brew coffee at home.

2.  **Determine Key Components of a Good Recipe:**
    *   Ingredients (Coffee beans/grind type, water ratio, optional sweeteners/flavorings).
    *   Equipment needed (Jar/Container, strainer/filter, fridge/freezer option).
    *   Step-by-step instructions (Grinding -> Mixing -> Steeping -> Straining -> Serving).
    *   Tips & Tricks (Storage, variations like milk or sugar).
    *   Safety/Cleanliness notes (Hygiene in brewing).

3.  **Drafting the Content (Iterative Process):**

    *   *Introduction:* Acknowledge what c

In [ ]:
# --- Hücre 7: Gerçek Eğitim (Tam Veri Seti - 2 Epoch) ---
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Eğitimi aktif et!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        num_train_epochs = 2, #
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # Vision fine-tuning için zorunlu ayarlar:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

# Eğitimi başlat
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 996 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,464,896 of 4,571,730,432 (0.71% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.817800
20,1.515789
30,1.304863
40,1.316328
50,1.261588
60,1.230598
70,1.226557
80,1.196435
90,1.175136
100,1.165134


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-250/tokenizer_config.json.


In [ ]:
# --- Hücre 8: Çıkarım (Test) ---
FastVisionModel.for_inference(model) # Inference hızlandırmasını aktif et

# Test için sorumuz
instruction = "Cold Brew Tarifi nasıl yapılır? Tarifini verebilir misin?"

messages = [
    {"role": "user", "content": [
        {"type": "text", "text": instruction}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    text=input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

print("\n--- Eğitilmiş Modelin Çıktısı ---\n")
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048,
                   use_cache = True, temperature = 0.7, min_p = 0.1,repetition_penalty=1.18)


--- Eğitilmiş Modelin Çıktısı ---

Kullanıcı Cold Brew Tarifi tarifi istiyor. Tabii, işte nefis bir Cold Brew Tarifi tarifi!

⏱️ **Süre:** PT1H | 🍽️ **Porsiyon:** 3

**Malzemeler:**
- 2 su bardağı toz kahve (fincan ölçüsü)
- 4 litre soğuk su
- Filtre kağıdı veya filtre makası

**Yapılışı:**
1. Kahveyi ve suyu karıştırma kabına alıp iyice çırpalım.
2. Üzerine temiz bir bez örtüp buzdolabında en az 8 saat bekletelim.
3. Daha sonra süzüp servis edebilirsiniz. Afiyet olsun 😋🌸

Şimdiden afiyet olsun!
</think>

Tabii, işte nefis bir Cold Brew Tarifi tarifi!

⏱️ **Süre:** PT1H | 🍽️ **Porsiyon:** 3

**Malzemeler:**
- 2 su bardağı toz kahve (fincan ölçüsü)
- 4 litre soğuk su
- Filtre kağıdı veya filtre makası

**Yapılışı:**
1. Kahveyi ve suyu karıştırma kabına alıp iyice çırpalım.
2. Üzerine temiz bir bez örtüp buzdolabında en az 8 saat bekletelim.
3. Daha sonra süzüp servis edebilirsiniz. Afiyet olsun 😋🌸

Şimdiden afiyet olsun!<|im_end|>


In [ ]:
# --- Hücre 9: Modeli Hugging Face'e Yükleme ---
# HUGGING FACE KULLANICI ADINI BURAYA YAZ:
MODEL_REPO_ID = "nypgd/qwen3.5-4b-recipe-lora"

print("Eğitilmiş LoRA ağırlıkları Hugging Face'e yükleniyor...")
# Modeli ve tokenizer'ı push_to_hub ile yüklüyoruz
model.push_to_hub(MODEL_REPO_ID)
tokenizer.push_to_hub(MODEL_REPO_ID)

print(f"🎉 Model başarıyla {MODEL_REPO_ID} adresine yüklendi! Ödev tamamlandı.")

Eğitilmiş LoRA ağırlıkları Hugging Face'e yükleniyor...


README.md:   0%|          | 0.00/527 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 52.7kB /  130MB            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp4lyzgve_/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp4lyzgve_/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

🎉 Model başarıyla nypgd/qwen3.5-4b-recipe-lora adresine yüklendi! Ödev tamamlandı.
